# Playground — `<modelo>` (`<seu nome>`)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpeuzinCosta/kws-borda/blob/main/notebooks/experimentos/playground_modelo.ipynb)

**Notebook autossuficiente para experimentar à vontade.** Tudo está aqui dentro (dados, MFCC, modelo, treino,
avaliação e latência) e nada é importado das pastas do repositório. Dá para mudar qualquer linha sem quebrar o
código de ninguém.

Como usar:
1. Abra no Colab e faça **Arquivo → Salvar uma cópia no Drive** (ou copie para
   `notebooks/experimentos/<modelo>_<nome>.ipynb`). Ambiente de execução → Alterar tipo → GPU.
2. Escreva o seu modelo na seção **4. Modelo**, no lugar de `MeuModelo`.
3. Use `fracao < 1` nos hiperparâmetros para iterar rápido; para números de verdade, `fracao = 1.0`.
4. Quando a arquitetura estiver definida, siga a seção **10. Levando para o repositório**.

Para os números serem comparáveis com a bancada (`dados/`, `treino/`, `avaliacao/`), este notebook usa as mesmas
regras: o split oficial por locutor, o mesmo MFCC, a mesma montagem da tarefa de 12 classes e o mesmo loop de
treino (AdamW + OneCycle + label smoothing). O cache de MFCC também é o mesmo arquivo usado pela bancada.

## 1. Setup

No Colab, o Google Drive é montado: o `.tar.gz` do GSC, o cache de MFCC e os resultados ficam em
`MyDrive/kws-borda/` e sobrevivem ao fim da sessão. Os áudios são extraídos no disco local (`/content/data`),
porque ler 100 mil arquivos direto do Drive é muito lento.

In [ ]:
#@title Setup — rode esta célula primeiro
import os, sys, copy, json, random, tarfile, time, urllib.request
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import torch
import torchaudio
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.utils.flop_counter import FlopCounterMode
from tqdm.auto import tqdm

NO_COLAB = "google.colab" in sys.modules
if NO_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ.setdefault("KWS_DRIVE", "/content/drive/MyDrive/kws-borda")
    os.environ.setdefault("KWS_DATA", "/content/data")

#Fora do Colab: usa a raiz do repositório (pastas data/, cache/ e runs/ já estão no .gitignore)
RAIZ_LOCAL = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists()), Path.cwd())
DRIVE = Path(os.environ.get("KWS_DRIVE", RAIZ_LOCAL))
DADOS = Path(os.environ.get("KWS_DATA", DRIVE / "data"))    #áudios extraídos (disco local)
DOWNLOADS = DRIVE / "downloads"                             #o .tar.gz é baixado UMA vez
CACHE = DRIVE / "cache"                                     #MFCC já calculado (mesmo cache da bancada)
SAIDA = DRIVE / "runs" / "playground"                       #pesos e log das tentativas
for p in [DADOS, DOWNLOADS, CACHE, SAIDA]:
    p.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Colab: {NO_COLAB} | device: {device} | torch {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-'}")
print(f"dados: {DADOS}\ncache: {CACHE}\nsaída: {SAIDA}")

## 2. Hiperparâmetros

Um único dicionário controla o experimento. Os hiperparâmetros **da arquitetura** (canais, camadas, kernel...)
ficam na seção 4, junto do modelo.

MFCC: o padrão é 40 coeficientes × 101 janelas (janela de 30 ms, passo de 10 ms). Para a entrada 8x menor do
artigo Hello Edge, use `n_mfcc=10, janela_ms=40, passo_ms=20, center=False` (10 × 49).

In [ ]:
NOME_EXPERIMENTO = "meu_modelo_v1"      #aparece no log de tentativas (seção 9)

hyperparameters = dict(
    #Tarefa: "gsc35" (35 palavras) ou "gsc12" (10 comandos + _unknown_ + _silence_)
    tarefa="gsc35",
    #Fração do TREINO usada (1.0 = tudo). Validação e teste são sempre completos.
    fracao=0.2,
    #MFCC
    sample_rate=16000, n_mfcc=40, janela_ms=30, passo_ms=10, n_mels=40, f_min=20, f_max=7600, center=True,
    #"global": média/desvio por coeficiente medidos no treino | "por_amostra": cada áudio pela própria média/desvio
    normalizacao="global",
    spec_augment=True,
    #Treino
    epocas=10,
    batch_size=128,
    learning_rate=3e-3,
    weight_decay=0.01,
    label_smoothing=0.1,
    seed=0,
)

## 3. Dados — Google Speech Commands v2

### 3.1 Download e extração
2,3 GB. Na primeira vez baixa para o Drive; nas próximas sessões só extrai (1–2 min).

In [ ]:
URL_GSC = "https://storage.googleapis.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz"
RAIZ_GSC = DADOS / "gsc"

if not (RAIZ_GSC / "testing_list.txt").exists():
    tar = DOWNLOADS / "speech_commands_v0.02.tar.gz"
    if not tar.exists():
        print("baixando o GSC v2 (2,3 GB)...")
        urllib.request.urlretrieve(URL_GSC, f"{tar}.parcial")
        os.rename(f"{tar}.parcial", tar)       #só vira o nome final quando o download terminou
    print("extraindo...")
    RAIZ_GSC.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar) as t:
        t.extractall(RAIZ_GSC)
print("GSC em", RAIZ_GSC)

### 3.2 Classes e split

O split é o **oficial, por locutor** (`validation_list.txt` e `testing_list.txt`): o mesmo locutor nunca aparece
no treino e no teste.

Na tarefa `gsc12`, a classe `_unknown_` é um sorteio das outras 25 palavras e `_silence_` são trechos de 1 s
dos ruídos de fundo (volume a 10%). O sorteio usa semente fixa (42), igual à bancada.

In [ ]:
CLASSES_GSC = [
    "backward", "bed", "bird", "cat", "dog", "down", "eight", "five", "follow", "forward",
    "four", "go", "happy", "house", "learn", "left", "marvin", "nine", "no", "off",
    "on", "one", "right", "seven", "sheila", "six", "stop", "three", "tree", "two",
    "up", "visual", "wow", "yes", "zero",
]
COMANDOS_12 = ["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"]
TAREFAS = {"gsc35": CLASSES_GSC, "gsc12": COMANDOS_12 + ["_unknown_", "_silence_"]}
CLASSES = TAREFAS[hyperparameters["tarefa"]]
N_AMOSTRAS = 16000      #1 s a 16 kHz


def listar_split(raiz, tarefa):
    """Caminhos relativos ('classe/arquivo.wav') de treino/val/teste."""
    ler = lambda f: set((raiz / f).read_text().split())
    val, teste = ler("validation_list.txt"), ler("testing_list.txt")
    split = {"treino": [], "val": [], "teste": []}
    for c in CLASSES_GSC:
        for f in sorted(p.name for p in (raiz / c).iterdir()):
            rel = f"{c}/{f}"
            split["val" if rel in val else "teste" if rel in teste else "treino"].append(rel)
    if tarefa == "gsc35":
        return split

    #gsc12: 10 comandos + unknown (sorteado) + silence ('_silence_/<ruído>@<início>')
    rng = random.Random(42)
    ruidos = {f.name: sf.info(f).frames for f in sorted((raiz / "_background_noise_").glob("*.wav"))}
    for nome, arquivos in split.items():
        comandos = [r for r in arquivos if r.split("/")[0] in COMANDOS_12]
        outros = [r for r in arquivos if r.split("/")[0] not in COMANDOS_12]
        por_classe = len(comandos) // len(COMANDOS_12)
        unknown = rng.sample(outros, min(por_classe, len(outros)))
        silencio = []
        for _ in range(por_classe):
            arq = rng.choice(list(ruidos))
            silencio.append(f"_silence_/{arq}@{rng.randint(0, ruidos[arq] - N_AMOSTRAS)}")
        split[nome] = comandos + unknown + silencio
    return split


def rotulo(rel):
    """'cat/x.wav' -> índice da classe (vira _unknown_ na gsc12)."""
    palavra = rel.split("/")[0]
    if hyperparameters["tarefa"] == "gsc12" and palavra not in COMANDOS_12 and palavra != "_silence_":
        palavra = "_unknown_"
    return CLASSES.index(palavra)


split = listar_split(RAIZ_GSC, hyperparameters["tarefa"])
print(hyperparameters["tarefa"], {k: len(v) for k, v in split.items()}, "| classes:", len(CLASSES))

### 3.3 MFCC

Cada áudio de 1 s vira uma matriz `(n_mfcc, n_frames)`: para cada janela de tempo, coeficientes que resumem o
espectro na escala Mel. O MFCC de todo o dataset é calculado **uma vez** e salvo em cache.

In [ ]:
def criar_extrator(hp):
    n_fft = int(hp["sample_rate"] * hp["janela_ms"] / 1000)     #30 ms -> 480 amostras
    hop = int(hp["sample_rate"] * hp["passo_ms"] / 1000)        #10 ms -> 160 amostras
    return torchaudio.transforms.MFCC(
        sample_rate=hp["sample_rate"], n_mfcc=hp["n_mfcc"],
        melkwargs=dict(n_fft=n_fft, win_length=n_fft, hop_length=hop, n_mels=hp["n_mels"],
                       f_min=hp["f_min"], f_max=hp["f_max"], center=hp["center"]))


def calcular_n_frames(hp):
    n_fft = int(hp["sample_rate"] * hp["janela_ms"] / 1000)
    hop = int(hp["sample_rate"] * hp["passo_ms"] / 1000)
    return 1 + N_AMOSTRAS // hop if hp["center"] else 1 + (N_AMOSTRAS - n_fft) // hop


def ler_audio(rel):
    """Áudio do GSC ou trecho de silêncio -> tensor (16000,). Curto: completa com zeros; longo: corta no centro."""
    if rel.startswith("_silence_/"):
        arq, inicio = rel.split("/", 1)[1].split("@")
        wav, _ = sf.read(RAIZ_GSC / "_background_noise_" / arq, dtype="float32",
                         start=int(inicio), frames=N_AMOSTRAS)
        return 0.1 * torch.from_numpy(wav)
    wav = torch.from_numpy(sf.read(RAIZ_GSC / rel, dtype="float32")[0])
    if len(wav) < N_AMOSTRAS:
        return nn.functional.pad(wav, (0, N_AMOSTRAS - len(wav)))
    inicio = (len(wav) - N_AMOSTRAS) // 2
    return wav[inicio:inicio + N_AMOSTRAS]


def extrair_mfcc(arquivos, caminho_cache, extrator, batch=256):
    """MFCC de todos os arquivos (ou carrega do cache). Devolve (X, y)."""
    if caminho_cache.exists():
        return torch.load(caminho_cache)
    X, y = [], []
    with torch.no_grad():
        for i in tqdm(range(0, len(arquivos), batch), desc=caminho_cache.stem):
            lote = arquivos[i:i + batch]
            X.append(extrator(torch.stack([ler_audio(r) for r in lote])))
            y += [rotulo(r) for r in lote]
    X, y = torch.cat(X), torch.tensor(y)
    torch.save((X, y), caminho_cache)
    return X, y


hp = hyperparameters
N_FRAMES = calcular_n_frames(hp)
extrator = criar_extrator(hp)
#Mesmo nome de arquivo da bancada (dados/speech_commands.py): o cache serve para os dois
nome = f"{hp['tarefa']}_{hp['n_mfcc']}x{N_FRAMES}_{hp['janela_ms']}-{hp['passo_ms']}ms{'c' if hp['center'] else 'nc'}"
dados = {k: extrair_mfcc(v, CACHE / f"gsc_{k}_{nome}.pt", extrator) for k, v in split.items()}

#Fração do treino (para iterar rápido). Sorteio fixo: a mesma fração dá sempre os mesmos exemplos.
if hp["fracao"] < 1:
    X, y = dados["treino"]
    idx = torch.randperm(len(y), generator=torch.Generator().manual_seed(0))[: int(len(y) * hp["fracao"])]
    dados["treino"] = (X[idx], y[idx])

for k, (X, y) in dados.items():
    print(f"{k:6s} X: {tuple(X.shape)}  y: {tuple(y.shape)}")

In [ ]:
#Alguns exemplos de MFCC
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
X, y = dados["val"]
for ax, i in zip(axes, torch.randperm(len(y))[:4]):
    ax.imshow(X[i], aspect="auto", origin="lower", cmap="magma")
    ax.set_title(CLASSES[y[i]])
    ax.set_xlabel("frame (tempo)")
axes[0].set_ylabel("coeficiente MFCC")
plt.tight_layout()
plt.show()

### 3.4 Normalização, SpecAugment e DataLoaders

- **Normalização global:** média e desvio por coeficiente medidos **só no treino** (medir na validação seria
  vazamento).
- **SpecAugment** (só no treino): zera uma faixa de frames e uma de coeficientes, para a rede não depender de um
  único trecho do áudio.

In [ ]:
if hp["normalizacao"] == "global":
    MEDIA = dados["treino"][0].mean(dim=(0, 2), keepdim=True)[0]     #(n_mfcc, 1)
    DESVIO = dados["treino"][0].std(dim=(0, 2), keepdim=True)[0]
else:
    MEDIA = DESVIO = None


def normalizar(x):
    if MEDIA is None:
        return (x - x.mean(dim=-1, keepdim=True)) / (x.std(dim=-1, keepdim=True) + 1e-6)
    return (x - MEDIA) / DESVIO


def spec_augment(x, max_tempo=10, max_coef=7):
    x = x.clone()
    n_coef, n_frames = x.shape
    max_coef = min(max_coef, n_coef // 4)
    t = torch.randint(0, max_tempo + 1, ()).item()
    t0 = torch.randint(0, n_frames - t + 1, ()).item()
    x[:, t0:t0 + t] = 0
    f = torch.randint(0, max_coef + 1, ()).item()
    f0 = torch.randint(0, n_coef - f + 1, ()).item()
    x[f0:f0 + f, :] = 0
    return x


class DatasetMFCC(Dataset):
    def __init__(self, X, y, aumentar=False):
        self.X, self.y, self.aumentar = normalizar(X), y, aumentar

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        x = self.X[i]
        return (spec_augment(x) if self.aumentar else x), self.y[i]


loaders = {k: DataLoader(DatasetMFCC(X, y, aumentar=(k == "treino" and hp["spec_augment"])),
                         batch_size=hp["batch_size"], shuffle=(k == "treino"),
                         pin_memory=torch.cuda.is_available())
           for k, (X, y) in dados.items()}
xb, yb = next(iter(loaders["treino"]))
print("batch:", tuple(xb.shape), "| rótulos:", yb[:8].tolist())

## 4. Modelo

**Contrato** (o mesmo da bancada):
- entrada `x`: `(B, n_mfcc, n_frames)`, o MFCC normalizado;
- saída: logits `(B, n_classes)`;
- a última camada se chama `self.head`;
- o `__init__` recebe `n_classes, n_mfcc, n_frames` e, depois, os hiperparâmetros da arquitetura.

`MeuModelo` abaixo é só um exemplo mínimo para o notebook rodar de ponta a ponta. Substitua pela sua arquitetura.

In [ ]:
class MeuModelo(nn.Module):
    """Exemplo mínimo (achata o MFCC + 1 camada escondida). Troque pela sua arquitetura."""

    def __init__(self, n_classes, n_mfcc=40, n_frames=101, hidden=64):
        super().__init__()
        self.corpo = nn.Sequential(
            nn.Flatten(),                           #(B, n_mfcc, n_frames) -> (B, n_mfcc * n_frames)
            nn.Linear(n_mfcc * n_frames, hidden),
            nn.ReLU(),
        )
        self.head = nn.Linear(hidden, n_classes)    #última camada sempre com o nome `head`

    def forward(self, x):
        return self.head(self.corpo(x))


#Hiperparâmetros da arquitetura (vão para o __init__)
hp_modelo = dict(hidden=64)


def criar_modelo():
    return MeuModelo(len(CLASSES), hp["n_mfcc"], N_FRAMES, **hp_modelo)

### 4.1 Contrato e custo

- **Parâmetros:** tamanho em memória = parâmetros × 4 bytes (fp32) ou × 1 byte (int8).
- **MACs:** multiplicações + somas para classificar **um** áudio (1 MAC = 2 FLOPs). Contados em modo `train()`
  porque o kernel otimizado de atenção do PyTorch em `eval()` esconde operações do contador.

In [ ]:
def custo(model):
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    estava_treinando = model.training
    model.train()   #batch 2 por causa do BatchNorm
    with FlopCounterMode(display=False) as contador, torch.no_grad():
        model(torch.randn(2, hp["n_mfcc"], N_FRAMES, device=next(model.parameters()).device))
    model.train(estava_treinando)
    return dict(params=params, kb_fp32=params * 4 / 1024, kb_int8=params / 1024,
                macs=contador.get_total_flops() // 2 // 2)     #/2 pelo batch, /2 de FLOPs para MACs


model = criar_modelo()
x = torch.randn(4, hp["n_mfcc"], N_FRAMES)
saida = model.eval()(x)
assert saida.shape == (4, len(CLASSES)), f"saída {tuple(saida.shape)}, esperado (4, {len(CLASSES)})"
assert hasattr(model, "head"), "a última camada deve se chamar `head`"
c = custo(model)
print(f"entrada {tuple(x.shape)} -> saída {tuple(saida.shape)}")
print(f"parâmetros: {c['params']:,} | {c['kb_fp32']:.1f} KB fp32 | {c['kb_int8']:.1f} KB int8 | MACs: {c['macs']:,}")

In [ ]:
#Resumo camada a camada (opcional)
try:
    import torchinfo
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchinfo"], check=True)
    import torchinfo
torchinfo.summary(model, input_size=(2, hp["n_mfcc"], N_FRAMES), device="cpu")

## 5. Treino

Mesmas regras da bancada: AdamW, OneCycle (o learning rate sobe no começo e desce até o fim), entropia cruzada com
label smoothing, e os pesos da melhor época na validação são os que ficam.

In [ ]:
def fixar_semente(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def treinar_uma_epoca(model, dataloader, criterion, optimizer, scheduler):
    model.train()
    soma, total = 0.0, 0
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()       #o PyTorch acumula gradientes
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        scheduler.step()            #OneCycle avança a cada batch
        soma += loss.item() * len(y)
        total += len(y)
    return soma / total


@torch.no_grad()
def validar_uma_epoca(model, dataloader, criterion):
    model.eval()
    soma, acertos, total = 0.0, 0, 0
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        soma += criterion(logits, y).item() * len(y)
        acertos += (logits.argmax(1) == y).sum().item()
        total += len(y)
    return soma / total, acertos / total


fixar_semente(hp["seed"])
model = criar_modelo().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=hp["learning_rate"], weight_decay=hp["weight_decay"])
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=hp["learning_rate"],
                                                total_steps=hp["epocas"] * len(loaders["treino"]))
criterion = nn.CrossEntropyLoss(label_smoothing=hp["label_smoothing"])

hist = {"treino": [], "val": [], "acc": []}
melhor_acc, melhor_estado = -1.0, None
inicio_treino = time.time()
for epoca in range(1, hp["epocas"] + 1):
    t0 = time.time()
    perda_treino = treinar_uma_epoca(model, loaders["treino"], criterion, optimizer, scheduler)
    perda_val, acc_val = validar_uma_epoca(model, loaders["val"], criterion)
    hist["treino"].append(perda_treino)
    hist["val"].append(perda_val)
    hist["acc"].append(acc_val)
    if acc_val > melhor_acc:
        melhor_acc = acc_val
        melhor_estado = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"época {epoca:2d}/{hp['epocas']} | perda treino: {perda_treino:.4f} | perda val: {perda_val:.4f} | "
          f"acurácia val: {acc_val*100:.2f}% | {time.time() - t0:.0f}s")

model.load_state_dict(melhor_estado)
tempo_treino = time.time() - inicio_treino
print(f"melhor acurácia val: {melhor_acc*100:.2f}% (época {hist['acc'].index(melhor_acc) + 1}) | "
      f"treino: {tempo_treino/60:.1f} min")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
ax1.plot(hist["treino"], label="treino")
ax1.plot(hist["val"], label="val")
ax1.set_xlabel("época"); ax1.set_ylabel("perda"); ax1.legend()
ax2.plot([a * 100 for a in hist["acc"]])
ax2.set_xlabel("época"); ax2.set_ylabel("acurácia val (%)")
plt.tight_layout()
plt.show()

## 6. Avaliação no teste

- **Acurácia:** fração de acertos.
- **F1 macro:** média do F1 de cada classe; pesa todas as classes igualmente.
- **Matriz de confusão:** linha = classe verdadeira, coluna = classe prevista (normalizada por linha).

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score


@torch.no_grad()
def prever(model, dataloader):
    model.eval()
    preds, reais = [], []
    for x, y in dataloader:
        preds.append(model(x.to(device)).argmax(1).cpu())
        reais.append(y)
    return torch.cat(preds).numpy(), torch.cat(reais).numpy()


y_pred, y_true = prever(model, loaders["teste"])
acc_teste = (y_pred == y_true).mean()
f1_teste = f1_score(y_true, y_pred, average="macro")
print(f"teste | acurácia: {acc_teste*100:.2f}% | F1 macro: {f1_teste:.4f}")

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)), normalize="true")
fig, ax = plt.subplots(figsize=(10, 9) if len(CLASSES) > 12 else (6, 5))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=90)
ax.set_yticks(range(len(CLASSES)), CLASSES)
ax.set_xlabel("previsto"); ax.set_ylabel("verdadeiro")
plt.tight_layout()
plt.show()

#Pares mais confundidos (fora da diagonal)
pares = [(CLASSES[i], CLASSES[j], cm[i, j]) for i in range(len(CLASSES)) for j in range(len(CLASSES)) if i != j]
pd.DataFrame(sorted(pares, key=lambda p: -p[2])[:10], columns=["verdadeiro", "previsto", "fração"])

## 7. Latência (MFCC × rede)

Protocolo da hipótese (b): CPU com **1 thread**, **batch 1** (um comando por vez, como no dispositivo),
aquecimento antes de medir e mediana de várias repetições. O MFCC e a rede são medidos separadamente.

In [ ]:
def cronometrar(fn, repeticoes=300, aquecimento=30):
    for _ in range(aquecimento):
        fn()
    tempos = []
    for _ in range(repeticoes):
        t0 = time.perf_counter()
        fn()
        tempos.append((time.perf_counter() - t0) * 1e3)
    return float(np.median(tempos)), float(np.percentile(tempos, 95))


threads_antes = torch.get_num_threads()
torch.set_num_threads(1)
with torch.no_grad():
    modelo_cpu = copy.deepcopy(model).cpu().eval()
    ext_cpu = criar_extrator(hp).cpu().eval()
    wav = torch.randn(1, N_AMOSTRAS) * 0.1
    feat = ext_cpu(wav)
    mfcc_ms, mfcc_p95 = cronometrar(lambda: ext_cpu(wav))
    rede_ms, rede_p95 = cronometrar(lambda: modelo_cpu(feat))
torch.set_num_threads(threads_antes)

print(f"MFCC: {mfcc_ms:.3f} ms (p95 {mfcc_p95:.3f}) | rede: {rede_ms:.3f} ms (p95 {rede_p95:.3f}) | "
      f"MFCC/rede = {mfcc_ms / rede_ms:.2f}x")

## 8. Resumo da tentativa

In [ ]:
resumo = dict(
    experimento=NOME_EXPERIMENTO, data=f"{datetime.now():%Y-%m-%d %H:%M}",
    tarefa=hp["tarefa"], entrada=f"{hp['n_mfcc']}x{N_FRAMES}", fracao=hp["fracao"], epocas=hp["epocas"],
    acc_val=round(melhor_acc * 100, 2), acc_teste=round(acc_teste * 100, 2), f1_teste=round(f1_teste, 4),
    params=c["params"], kb_int8=round(c["kb_int8"], 1), macs=c["macs"],
    mfcc_ms=round(mfcc_ms, 3), rede_ms=round(rede_ms, 3), treino_min=round(tempo_treino / 60, 1),
    hp_modelo=json.dumps(hp_modelo),
)
pd.DataFrame([resumo]).T.rename(columns={0: ""})

## 9. Salvar e comparar tentativas

Salva os pesos e acrescenta uma linha em `runs/playground/log.csv` (no Drive), para comparar as tentativas entre
sessões.

In [ ]:
arquivo = SAIDA / f"{NOME_EXPERIMENTO}_{datetime.now():%Y%m%d-%H%M%S}.pt"
torch.save({"estado": model.state_dict(), "hyperparameters": hp, "hp_modelo": hp_modelo,
            "hist": hist, "resumo": resumo}, arquivo)

LOG = SAIDA / "log.csv"
log = pd.concat([pd.read_csv(LOG), pd.DataFrame([resumo])]) if LOG.exists() else pd.DataFrame([resumo])
log.to_csv(LOG, index=False)
print("salvo em", arquivo)
log.tail(10)

## 10. Levando para o repositório

Quando a arquitetura estiver definida:
1. Copie a classe para `modelos/<modelo>.py`, no lugar do esqueleto, com o decorador `@registrar_modelo(...)` que
   já está lá, e mude `IMPLEMENTADO = True`.
2. Passe os valores de `hp_modelo` para `configs/<modelo>.yaml` (seção `modelo:`), e o que mudou em
   `hyperparameters` (épocas, learning rate...) também.
3. Rode `pytest` para conferir o contrato.
4. Rode o `template_modelo.ipynb` com o seu modelo e confira se os números batem com os daqui (com `fracao = 1.0`).

## Anotações

| Data | Mudança | Params | Acurácia val | Observação |
|---|---|---|---|---|
| | | | | |